# Energy Calibration
_Author: Danaé Valdenaire, Philipp Schreiner_<br>
_Created: Sep. 5, 2025_<br>
_Last updated: May. 5th, 2026 by Danaé Valdenaire_

---

## Introduction

This tutorial walks you through the **energy calibration** using the CAIT software. But first, a little bit of context is required.

The detector response varies over time due to temperature fluctuations and other instabilities. 
As a result, the **pulse height (PH)** recorded for a given event does not directly reflect the deposited energy. To account for these variations, **testpulses** of known amplitude are injected periodically. By tracking their reconstructed amplitude (**TPA**) over time, we build a model of how the detector response changes. This model is then used to map each event's pulse height to a **testpulse-equivalent amplitude (TPE)** — a corrected quantity that can finally be converted into a physical energy using calibration sources.

```{important}
**Testpulse Response:**

- [**`vai.TPRPoly`**](cait.versatile.TPRPoly): models the testpulse amplitude 
  over time with a **polynomial**. Use this when the detector drift is **gradual 
  and regular** throughout the run.

- [**`vai.TPRCubicSpline`**](cait.versatile.TPRCubicSpline): models the testpulse 
  amplitude over time with a **cubic spline**. Use this when the detector drift 
  shows **sudden or irregular changes**, as the spline can adapt to local variations 
  better than a polynomial.

**Transfer Function:**

- [**`vai.TFPchip`**](cait.versatile.TFPchip): builds the transfer function using 
  a **piecewise cubic interpolation**. Use this when the amplitude correction is 
  **not perfectly linear** across all testpulse amplitudes, as it connects the 
  control points smoothly without introducing unwanted oscillations.

- [**`vai.TFPoly`**](cait.versatile.TFPoly): builds the transfer function using 
  a **polynomial fit**. Use this when the amplitude correction is **close to 
  linear** and a simple, smooth function is sufficient.
```

```{tip}
`vai.TPRCubicSpline` + `vai.TFPchip` is the most robust combination when the detector response is not perfectly stable. 
```


## First, you need data

This part is from the triggering stream data tutorial.

In [ ]:
# One hour of stream data with two channels:
# Here, we set a random seed so that the tutorial looks the same for you.
# If you want to trigger actual data, you want to construct either of the
# stream objects explained above, depending on the hardware used.
stream = vai.MockStream(seed=137, rate_Hz=2)

# You can check which channels are present in the stream ...
print(f"Available channels: {stream.keys}")
# ... which testpulse channels are available ...
print(f"Available TP channels: {stream.tp_keys}")
# ... and which TPAs are available:
print(f"Available TPAs:", {k: np.unique(v) for k, v in stream.tpas.items()})

# Just putting the stream object at the end of a cell gives you 
# an information overview. E.g. the timebase 'dt_us' and the 
# measuring time in hours 'measuring_time_h'.
stream

In [ ]:
# Configure trigger
record_length = 2**14

# Basic configuration
trigger_config = {
    "trigger_channels": ["Ch0"], # those will be triggered, e.g. phonon channel
    "passive_channels": ["Ch1"], # those will be read in coincidence, e.g. light channel
    "testpulse_channels": ["TP0", "TP1"], # the testpulse channels corresponding to all trigger/passive channels
    "controlpulses_above": [9., 9.], # controlpulses have TPA=10 (see above)
    "f_noise": 1000, # we will sample 1000 random noise traces per hour (later needed for NPS creation)
    "copy_events": True, # set this to False if you don't want to copy the raw data of the stream to the HDF5 file (to save disk space)
}

In [ ]:
# Path to where we want to save the HDF5 file
fdirh5 = "tutorial_output"
hdf5_name = "my_first_trigger"

os.makedirs(fdirh5, exist_ok=True)

dh = ai.DataHandler(record_length=record_length, 
                    nmbr_channels=len(trigger_config["trigger_channels"]) + len(trigger_config["passive_channels"]), 
                    sample_frequency=stream.sample_frequency)
dh.set_filepath(path_h5=fdirh5, 
                fname=hdf5_name, 
                appendix=False)
dh.init_empty()
print(dh)

In [ ]:
dh.trigger_zscore(stream, **trigger_config)

In [ ]:
for group in ["events", "testpulses", "noise", "controlpulses"]:
    dh.cmp(group)

## Cleaning Testpulses

Before building the testpulse response, we need to remove corrupted or unstable 
testpulses. We apply several quality cuts on the testpulse parameters. The goal 
is to keep only testpulses that are well-reconstructed and representative of the 
detector's response.

```{tip}
You can perform a template or parametric fit on your testpulses 
beforehand using the function [**`vai.TemplateFit`**](`cait.versatile.TemplateFit`). This could be 
helpful for the next step. 

```

We will apply a stability cut and some quality cuts on our testpulses. The goal 
at this step is to obtain a clear behavior of each testpulse amplitude distribution 
over time. Each testpulse amplitude should look like a line (or a band) for the 
fit to account for the detector response as closely as possible.


In [ ]:
# Example on how to perform a stability cut:

tp_tpa = dh["testpulses/testpulseamplitude", 0]
tp_ph = dh["testpulses/parametric_fit", 0]
unique_tp = np.unique(tp_tpa)
print('Unique test pulse heights: ', unique_tp)

# For each TPA value, the median and the one sigma upper and lower limit are calculated.
medians = [np.median(tp_ph[tp_tpa == tpa]) for tpa in unique_tp]
lower_quantiles = [np.quantile(tp_ph[tp_tpa == tpa], 0.00135) for tpa in unique_tp]
upper_quantiles = [np.quantile(tp_ph[tp_tpa == tpa], 0.99865) for tpa in unique_tp]
mean_deviations = [(u - l) for l,u in zip(lower_quantiles, upper_quantiles)]

lb = [ l for l,m in zip(lower_quantiles, mean_deviations)]
ub = [ u for u,m in zip(upper_quantiles, mean_deviations)]

dh.calc_testpulse_stability(channel=0, significance=3, ub = ub, lb = lb, noise_level=0.003) 

For the quality cuts, as you probably already seen in the notebook on *sev*, there is no general rules. The best way to do it is to look at your data and try what works well. You can apply cuts on all the parameters in your `testpulses` in your data handler. If you performed a template or parametric fit on your testpulses, you can also apply an upper limit on the rms of the fit `dh["testpulses/rms_param_fit", 0]`. This is a good way to remove artefacts.  

In [ ]:
cut_rms_fit = (dh["testpulses/rms_param_fit",0]<0.03)
cut_neg_ph = (dh["testpulses/amp_param_fit",0]>0)
cut_var_first = (dh["testpulses/var_first_eight",0]<250e-6)
cut_slope = (dh["testpulses/slope",0]<250e-6)

tp_stability = dh["testpulses/testpulse_stability",0] #this flag should appear in your dh after running dh.calc_testpulse_stability

tp_quality_cuts = cut_rms_fit*cut_neg_ph*cut_var_first*cut_slope*tp_stability

To evaluate the quality of your cuts, you could plot the test pulses over time before and after your cuts. With the function [**`vai.ScatterPreview`**](`cait.versatile.ScatterPreview`) you can click on each point of the scatter plot to visualise each event. 

In [ ]:
vai.ScatterPreview(x=dh["testpulses/hours"][tp_quality_cuts], y=dh["testpulses/new_trunc_fit",0][tp_quality_cuts],
                    ev_it=dh.get_event_iterator("testpulses", channel=0, flag=tp_quality_cuts).with_processing(vai.RemoveBaseline()),
                    xlabel="Time (h)",
                    ylabel="Test pulse amplitude from parametric fit")

When you are happy with all your cleaning, you can move on to the calibration part $\downarrow$

## Evaluating Calibration

In [ ]:
# Load the testpulse timestamps, tpa and amplitudes from the parametric fit

tp_ts = dh.get_event_iterator("testpulses", channel=0).timestamps[tp_quality_cuts] 
tpas = dh["testpulses/testpulseamplitude",0][tp_quality_cuts]
tp_phs = dh["testpulses/amp_param_fit",0][tp_quality_cuts]

In [ ]:
# quick and dirty to remove outliers

cond = np.ones(len(tpas), dtype=bool)
for tpa in unique_tp:
    q0, q1, q2 = np.quantile(tp_phs[tpas==tpa], sp.stats.norm.cdf([-0.5,0,0.5]))
    cond[tpas==tpa] = abs((tp_phs[tpas==tpa]-q1)/(q2-q0))<3
    
tp_ts = tp_ts[cond]
tpas = tpas[cond]
tp_phs = tp_phs[cond]

Now, we are all set to perform the energy calibration with the function `cait.versatile.EnergyCalibration`. At this stage, We will give as inputs the testpulse amplitude over time: `tp_x=tp_ts` and `tp_phs=tp_phs`. We also have to choose the `testpulse_response`and the `transfer_function` described in the first section. You can click at any time in the left plot to see what the transfer function looks like at that time.

In [ ]:
my_ecal = vai.EnergyCalibration(tp_x=tp_ts,
                                tp_phs=tp_phs,
                                tpas=tpas,
                                testpulse_response=vai.TPRCubicSpline(kernel_length=1,remove_outliers=True),
                                transfer_function=vai.TFPchip(fix_at_yaxis=False),
                                max_x_gap=0.5)

With the preview, you can click at any time in the left plot to see what the transfer function looks like at that time. this makes you able to build the transfer function in an interactive way to find the combination that work the best to describe your data.  

In [ ]:
my_ecal.preview() 

```{important}
`my_ecal` is the object that you will use to tranform pulse heights in testpulse equivalent and vice-versa. 
- `my_ecal()`: pulse height $\rightarrow$ testpulse equivalent.
- `my_ecal.inverse()`: pulse height $\leftarrow$ testpulse equivalent.
```

```{tip} 
If you have a doubt on whether you have to use the normal or the inverse, you can print the argument of `my_ecal()` or `my_ecal.inverse()` using `tab`. One function takes the pulse height as an argument, the other the testpulse response.
```

- For instance, if you want to compute the testpulse equivalent you need to do $\downarrow$

In [ ]:
event_ts = dh.get_event_iterator("events", channel=0).timestamps
event_phs = dh["events/amp_param_fit",0]

TPE = my_ecal.inverse(event_ts, event_phs)
dh.set("events", TPE=TPE, overwrite_existing=True) # don't forget to save your calculation in your data handler.

In the other direction, it would be:

In [ ]:
output_phs = my_ecal(event_ts, tpes_of_interest)

When you are done with calibration, don't forget to save your calibration function to a file $\downarrow$

In [ ]:
my_ecal.to_file("my_ecal_function")

And if you want to load it later, do $\downarrow$

In [ ]:
my_ecal = vai.EnergyCalibration.from_file("my_ecal_function")

## Advanced

Give an overview of how people can implement new transfer functions and testpulse responses

```{warning}
WORK IN PROGRESS
```